<a href="https://colab.research.google.com/github/saranyakandasamy2003/BloomingFlower/blob/main/third_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np

# --- Conceptual Framework for Many-Objective Jaccard-Based Evolutionary Feature Selection ---

class Individual:
    """ Represents an individual (a subset of features) in the evolutionary algorithm. """
    def __init__(self, num_features):
        # A binary vector where 1 means feature is selected, 0 means not selected
        self.features = [random.randint(0, 1) for _ in range(num_features)]
        self.objectives = [] # To store fitness values (e.g., accuracy, number of features)
        self.rank = None
        self.crowding_distance = None
        self.num_features = num_features # Store num_features as an instance variable

    def __repr__(self):
        return (f"Individual(Features: {sum(self.features)}/{len(self.features)}, "
                f"Objs: {self.objectives}, Rank: {self.rank}, CD: {self.crowding_distance:.2f})")

    def calculate_objectives(self, data_placeholder=None):
        """
        This is where the 'Jaccard-Based' fitness and other objectives would be implemented.

        In a real scenario, this would involve:
        1. Using the selected features (self.features) to subset the actual dataset.
        2. Calculating Jaccard-based similarity or other relevant metrics (as described in the PDF).
        3. Training a classifier (e.g., SVM, Decision Tree) on the subsetted data.
        4. Evaluating performance (e.g., F1-score for imbalanced data, accuracy, precision, recall).
        5. Also, typically, the number of selected features is an objective (to minimize it).

        Since we don't have a dataset, we'll use placeholder objectives.
        """
        num_selected_features = sum(self.features)
        if num_selected_features == 0: # Avoid division by zero if no features are selected
            # Assign very poor objective values for individuals with no features
            self.objectives = [0.0, self.num_features]
            return

        # These are dummy objective values. The paper would define specific objectives.
        # Objective 1: Maximize 'classification performance' (higher is better)
        # Objective 2: Minimize 'number of selected features' (lower is better)

        # Example placeholder objectives:
        # Dummy performance: higher with more features, but with some randomness
        dummy_performance = 0.5 + (num_selected_features / self.num_features) * 0.4 + random.uniform(-0.05, 0.05)
        dummy_performance = max(0.0, min(1.0, dummy_performance)) # Clamp between 0 and 1

        self.objectives = [dummy_performance, num_selected_features]


def dominates(ind1, ind2, num_objectives):
    """ Checks if individual ind1 dominates individual ind2. """
    # For this conceptual example, we assume objective 0 is to be maximized, others minimized.
    # In a real MOEA, objectives can be specified as min/max.
    # Paper implies: Maximize classification performance (Obj0), Minimize features (Obj1)

    better_in_at_least_one = False
    for i in range(num_objectives):
        if i == 0: # Objective 0 is performance (maximize)
            if ind1.objectives[i] < ind2.objectives[i]: return False
            if ind1.objectives[i] > ind2.objectives[i]: better_in_at_least_one = True
        else: # Other objectives (e.g., number of features) are to be minimized
            if ind1.objectives[i] > ind2.objectives[i]: return False
            if ind1.objectives[i] < ind2.objectives[i]: better_in_at_least_one = True
    return better_in_at_least_one

def fast_non_dominated_sort(population, num_objectives):
    """
    Performs fast non-dominated sorting as in NSGA-II.
    Returns a list of fronts, where each front contains individuals.
    """
    S = [[] for _ in population]  # Set of individuals dominated by p
    n = [0 for _ in population]   # Dominating count for p
    fronts = [[]]

    for p_idx, p in enumerate(population):
        for q_idx, q in enumerate(population):
            if p_idx == q_idx: continue

            # Ensure objectives are calculated before comparison
            if not p.objectives: p.calculate_objectives()
            if not q.objectives: q.calculate_objectives()

            if dominates(p, q, num_objectives):
                S[p_idx].append(q)
            elif dominates(q, p, num_objectives):
                n[p_idx] += 1

        if n[p_idx] == 0:
            p.rank = 0
            fronts[0].append(p)

    i = 0
    while fronts[i]:
        Q = [] # next front
        for p in fronts[i]:
            for q in S[population.index(p)]: # Find index of p in original population
                n[population.index(q)] -= 1
                if n[population.index(q)] == 0:
                    q.rank = i + 1
                    Q.append(q)
        i += 1
        fronts.append(Q)

    return [front for front in fronts if front] # Filter out empty fronts

def calculate_crowding_distance(front, num_objectives):
    """
    Calculates crowding distance for individuals within a given front.
    Higher crowding distance means the individual is in a less crowded region.
    """
    if not front:
        return

    if len(front) <= 2: # Edge case for small fronts
        for ind in front:
            ind.crowding_distance = float('inf')
        return

    for ind in front: # Initialize crowding distance
        ind.crowding_distance = 0.0

    for m in range(num_objectives):
        # Sort front by objective m
        # For objective 0 (performance), sort descending (maximize). For others, ascending (minimize).
        if m == 0:
            front.sort(key=lambda x: x.objectives[m], reverse=True)
        else:
            front.sort(key=lambda x: x.objectives[m])

        # Set boundary points to infinity
        front[0].crowding_distance = float('inf')
        front[-1].crowing_distance = float('inf')

        obj_min = front[-1].objectives[m]
        obj_max = front[0].objectives[m]

        if obj_max == obj_min: # Handle case where all individuals have same value for this objective
            continue

        for i in range(1, len(front) - 1):
            front[i].crowding_distance += (front[i-1].objectives[m] - front[i+1].objectives[m]) / (obj_max - obj_min)

def initialize_population(pop_size, num_features):
    """ Initializes a population of individuals. """
    return [Individual(num_features) for _ in range(pop_size)]

def crossover(parent1, parent2):
    """ Performs crossover to create new offspring using a single-point crossover. """
    num_features = len(parent1.features)
    if num_features < 2: # Crossover not possible for less than 2 features
        return Individual(num_features), Individual(num_features)

    crossover_point = random.randint(1, num_features - 1)

    child1 = Individual(num_features)
    child1.features = parent1.features[:crossover_point] + parent2.features[crossover_point:]

    child2 = Individual(num_features)
    child2.features = parent2.features[:crossover_point] + parent1.features[crossover_point:]

    return child1, child2

def mutate(individual, mutation_rate):
    """ Mutates an individual's features by flipping bits with a given probability. """
    for i in range(len(individual.features)):
        if random.random() < mutation_rate:
            individual.features[i] = 1 - individual.features[i] # Flip the bit


def evolve(pop_size, num_features, generations, mutation_rate, num_objectives=2):
    """ Main evolutionary loop implementing conceptual NSGA-II-like selection. """
    population = initialize_population(pop_size, num_features)

    for individual in population:
        individual.calculate_objectives() # Calculate initial objectives

    for generation in range(generations):
        print(f"\n--- Generation {generation + 1}/{generations} ---")

        # Create offspring through selection, crossover, and mutation
        offspring_population = []
        while len(offspring_population) < pop_size: # Generate 'pop_size' offspring
            # Simplified tournament selection for parents (could be more complex)
            parent1 = random.choice(population)
            parent2 = random.choice(population)

            child1, child2 = crossover(parent1, parent2)
            mutate(child1, mutation_rate)
            mutate(child2, mutation_rate)

            offspring_population.append(child1)
            if len(offspring_population) < pop_size:
                offspring_population.append(child2)

        # Calculate objectives for all offspring
        for ind in offspring_population:
            ind.calculate_objectives()

        # Combine parent and offspring populations
        combined_population = population + offspring_population

        # Perform Fast Non-Dominated Sort
        fronts = fast_non_dominated_sort(combined_population, num_objectives)

        new_population = []
        current_front_idx = 0

        # Select individuals for the next generation based on fronts and crowding distance
        while len(new_population) + len(fronts[current_front_idx]) <= pop_size:
            calculate_crowding_distance(fronts[current_front_idx], num_objectives)
            new_population.extend(fronts[current_front_idx])
            current_front_idx += 1
            if current_front_idx >= len(fronts): # Ran out of fronts
                break

        # If the last front is too large, use crowding distance to select individuals
        if current_front_idx < len(fronts):
            remaining_individuals = fronts[current_front_idx]
            calculate_crowding_distance(remaining_individuals, num_objectives)
            # Sort by crowding distance in descending order (higher is better)
            remaining_individuals.sort(key=lambda x: x.crowding_distance, reverse=True)
            new_population.extend(remaining_individuals[:pop_size - len(new_population)])

        population = new_population

        # Print example individuals from the current population (top of the first front)
        print(f"Sample of Front 1 individuals (Objectives: [Performance (Max), Num_Features (Min)]):")
        for i in range(min(5, len(population))):
            if population[i].objectives:
                print(f"  Ind {i+1} (R{population[i].rank}, CD:{population[i].crowding_distance:.2f}): "
                      f"Selected Features: {sum(population[i].features)}, "
                      f"Objs: Perf={population[i].objectives[0]:.2f}, NumFeat={population[i].objectives[1]}")

    print("\n--- Final Population (Approximated Pareto Front) ---")
    # The final population represents the approximation of the Pareto front
    # Sort by rank, then crowding distance for display
    population.sort(key=lambda x: (x.rank, -x.crowding_distance) if x.rank is not None else (float('inf'), 0))

    for ind in population:
        if ind.objectives: # Ensure objectives are calculated
            print(f"Rank:{ind.rank}, CD:{ind.crowding_distance:.2f}, "
                  f"Features selected: {sum(ind.features)}, "
                  f"Objs: Performance={ind.objectives[0]:.2f}, Num_Features={ind.objectives[1]}")

# --- Parameters (conceptual) ---
NUM_TOTAL_FEATURES = 100 # Total number of features in your dataset (e.g., from the high-dimensional data)
POPULATION_SIZE = 50
GENERATIONS = 20
MUTATION_RATE = 0.1
NUM_OBJECTIVES = 2 # Assuming performance and number of features

print(f"Starting Many-Objective Evolutionary Feature Selection (Conceptual) with {NUM_TOTAL_FEATURES} features...")
evolve(POPULATION_SIZE, NUM_TOTAL_FEATURES, GENERATIONS, MUTATION_RATE, NUM_OBJECTIVES)

print("\n--- IMPORTANT NOTES --- ")
print("1. This is a *conceptual* template implementing the *structure* of an NSGA-II-like algorithm.")
print("2. The 'calculate_objectives' method still uses *placeholder* values for performance. To make this functional, \n   you must replace these with actual calculations based on a dataset and the 'Jaccard-based' fitness \n   functions and other objectives described in the PDF.")
print("3. The `dominates`, `fast_non_dominated_sort`, and `calculate_crowding_distance` functions are simplified \n   conceptual implementations. For production use, consider established MOEA libraries like `pymoo` or `Platypus`.")
print("4. Without a dataset, this code demonstrates the *flow* and *logic* of the algorithm, but not its \n   functional performance on real data. It will show how individuals evolve through generations \n   and get ranked/sorted by the MOEA principles.")

Starting Many-Objective Evolutionary Feature Selection (Conceptual) with 100 features...

--- Generation 1/20 ---
Sample of Front 1 individuals (Objectives: [Performance (Max), Num_Features (Min)]):
  Ind 1 (R0, CD:inf): Selected Features: 38, Objs: Perf=0.65, NumFeat=38
  Ind 2 (R0, CD:0.57): Selected Features: 39, Objs: Perf=0.70, NumFeat=39
  Ind 3 (R0, CD:0.26): Selected Features: 42, Objs: Perf=0.71, NumFeat=42
  Ind 4 (R0, CD:0.17): Selected Features: 43, Objs: Perf=0.72, NumFeat=43
  Ind 5 (R0, CD:0.16): Selected Features: 44, Objs: Perf=0.72, NumFeat=44

--- Generation 2/20 ---
Sample of Front 1 individuals (Objectives: [Performance (Max), Num_Features (Min)]):
  Ind 1 (R0, CD:inf): Selected Features: 34, Objs: Perf=0.59, NumFeat=34
  Ind 2 (R0, CD:0.42): Selected Features: 37, Objs: Perf=0.62, NumFeat=37
  Ind 3 (R0, CD:0.49): Selected Features: 38, Objs: Perf=0.65, NumFeat=38
  Ind 4 (R0, CD:0.40): Selected Features: 39, Objs: Perf=0.70, NumFeat=39
  Ind 5 (R0, CD:0.21): Sele